# 03 — Seq2Seq with Attention: Toy Translation Example (PyTorch)

This notebook builds a minimal encoder-decoder model with attention, trained on a tiny toy "translation" task (reversing a sequence of digits — a common Seq2Seq sanity-check task). The point is to see the encoder-decoder-attention architecture clearly, without the complexity of real language data.

**What you'll see:**
- An encoder RNN that compresses an input sequence
- A decoder RNN that generates an output sequence step-by-step
- A simple attention mechanism connecting the two

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

torch.manual_seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 1. Toy task: reverse a sequence of digits

Input: `[3, 1, 4, 2]` → Target: `[2, 4, 1, 3]`

This is a standard sanity-check task for Seq2Seq models because it requires the decoder to attend to different input positions depending on the output step — exactly what attention is for.

In [ ]:
SOS, EOS, PAD = 10, 11, 12  # special tokens
VOCAB_SIZE = 13  # digits 0-9 + SOS/EOS/PAD
SEQ_LEN = 6

def make_example():
    seq = [random.randint(0, 9) for _ in range(SEQ_LEN)]
    src = seq + [EOS]
    tgt = [SOS] + seq[::-1] + [EOS]
    return src, tgt

def make_batch(batch_size):
    srcs, tgts = zip(*[make_example() for _ in range(batch_size)])
    return torch.tensor(srcs).to(device), torch.tensor(tgts).to(device)

src_example, tgt_example = make_example()
print(f"Source: {src_example}")
print(f"Target: {tgt_example}")

## 2. Encoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.gru(embedded)  # outputs: all timesteps, hidden: final state
        return outputs, hidden

## 3. Attention module

A simple Bahdanau-style additive attention: score each encoder output against the current decoder hidden state, softmax to get weights, then take a weighted sum.

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: (batch, hidden_dim) -> expand to match encoder_outputs seq_len
        seq_len = encoder_outputs.size(1)
        decoder_hidden_expanded = decoder_hidden.unsqueeze(1).repeat(1, seq_len, 1)

        energy = torch.tanh(self.attn(torch.cat([decoder_hidden_expanded, encoder_outputs], dim=2)))
        scores = self.v(energy).squeeze(2)          # (batch, seq_len)
        weights = F.softmax(scores, dim=1)           # attention weights

        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)  # weighted sum
        return context, weights

## 4. Decoder

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = Attention(hidden_dim)
        self.gru = nn.GRU(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_token, hidden, encoder_outputs):
        embedded = self.embedding(input_token).unsqueeze(1)          # (batch, 1, embed_dim)
        context, weights = self.attention(hidden.squeeze(0), encoder_outputs)
        gru_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)
        output, hidden = self.gru(gru_input, hidden)
        logits = self.fc(output.squeeze(1))
        return logits, hidden, weights

## 5. Training loop

In [ ]:
encoder = Encoder(VOCAB_SIZE).to(device)
decoder = Decoder(VOCAB_SIZE).to(device)

optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

def train_step(batch_size=32):
    src, tgt = make_batch(batch_size)
    encoder_outputs, hidden = encoder(src)

    loss = 0
    decoder_input = tgt[:, 0]  # SOS token for every example

    for t in range(1, tgt.size(1)):
        logits, hidden, _ = decoder(decoder_input, hidden, encoder_outputs)
        loss += criterion(logits, tgt[:, t])
        decoder_input = tgt[:, t]  # teacher forcing: feed the true previous token

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(decoder.parameters()), 1.0)
    optimizer.step()

    return loss.item() / tgt.size(1)

for step in range(1, 1001):
    loss = train_step()
    if step % 100 == 0:
        print(f"Step {step}/1000 - Loss: {loss:.4f}")

## 6. Try inference on a new sequence

In [ ]:
def translate(seq):
    encoder.eval()
    decoder.eval()
    src = torch.tensor([seq + [EOS]]).to(device)

    with torch.no_grad():
        encoder_outputs, hidden = encoder(src)
        decoder_input = torch.tensor([SOS]).to(device)
        result = []

        for _ in range(len(seq) + 1):
            logits, hidden, _ = decoder(decoder_input, hidden, encoder_outputs)
            next_token = torch.argmax(logits, dim=1)
            if next_token.item() == EOS:
                break
            result.append(next_token.item())
            decoder_input = next_token

    return result

test_seq = [5, 8, 1, 9, 3, 2]
print(f"Input:    {test_seq}")
print(f"Expected: {test_seq[::-1]}")
print(f"Model output: {translate(test_seq)}")

## Try it yourself

- Print the attention `weights` returned by the decoder at each step and visualize them as a heatmap — you should see the model attending to later input positions first (since it's reversing the sequence).
- Replace teacher forcing with the model's own predictions during training (scheduled sampling) and see how training stability changes.
- Try a real toy translation dataset (e.g., small English-French phrase pairs) instead of digit reversal.
- Compare this attention-based decoder against a decoder with no attention (just the final encoder hidden state) — does removing attention hurt performance on longer sequences?